# HDPM_097 Assignment 2

## Iterative development of simulation model

The aim of this experiment is to see if a LLM can be used to replicate the stroke pathway simulation model described by Monks et Al (2016).

## Iteration 2

The prompt and output are available at: https://chatgpt.com/share/69a47680-0fc0-800c-a25a-1e6cd2ff602c

## Prompt

Please make the following refinements to the code you have generated:

1. A Scenario class. It is good practice to pass all of the parameters to the simulation model in a container. A class is a flexible way to achieve this aim. Hard code the data representing the base case. mMakes use of the default parameters to set up the base case scenario. 

2. Extend the sampling helper function use distribution classes. Define the following classes: Exponential and Lognormal. It is important to be able to pass random_seed when instantiating a new distribution instance.

3. Add a utility function trace(msg), for selectively printing the output of simulation in the console. Set a TRACE constant to FALSE to turn printing off, and TRUE to turn printing on.


In [4]:
"""
Stroke pathway DES (Iteration 1): arrivals only

Python: 3.11
SimPy:  4.1.1

Refinements included:
1) Scenario container class (base case hard-coded via defaults)
2) Distribution classes: Exponential, Lognormal (seedable)
3) trace(msg) utility controlled by TRACE constant
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Optional, Sequence

import simpy
import numpy as np


# -------------------------
# Tracing utility
# -------------------------

TRACE = True  # set True to print simulation trace output


def trace(msg: str) -> None:
    """Conditionally print trace messages."""
    if TRACE:
        print(msg)


# -------------------------
# Distribution classes
# -------------------------

class Distribution:
    """Base class for seedable distributions."""
    def __init__(self, random_seed: Optional[int] = None) -> None:
        self.rng = np.random.default_rng(random_seed)


class Exponential(Distribution):
    """Exponential distribution parameterised by mean (scale)."""
    def __init__(self, mean: float, random_seed: Optional[int] = None) -> None:
        if mean <= 0:
            raise ValueError("Exponential mean must be > 0")
        super().__init__(random_seed=random_seed)
        self.mean = float(mean)

    def sample(self) -> float:
        return float(self.rng.exponential(scale=self.mean))


class Lognormal(Distribution):
    """
    Lognormal distribution parameterised by mu, sigma (of underlying normal).
    Note: not used in iteration 1 arrivals, but provided for later pathway stages.
    """
    def __init__(self, mu: float, sigma: float, random_seed: Optional[int] = None) -> None:
        if sigma <= 0:
            raise ValueError("Lognormal sigma must be > 0")
        super().__init__(random_seed=random_seed)
        self.mu = float(mu)
        self.sigma = float(sigma)

    def sample(self) -> float:
        return float(self.rng.lognormal(mean=self.mu, sigma=self.sigma))


# -------------------------
# Scenario container
# -------------------------

@dataclass
class Scenario:
    """
    Container for model inputs.

    Base-case values are hard-coded via defaults.
    """
    random_seed: int = 42

    # Category names (keep stable order for reporting / reproducibility)
    categories: tuple[str, ...] = ("stroke", "tia", "complex_neuro", "other")

    # Base-period counts (for reference / checking proportions)
    base_period_counts: dict[str, int] = field(default_factory=lambda: {
        "stroke": 1320,
        "tia": 158,
        "complex_neuro": 456,
        "other": 510,
    })

    # Mean inter-arrival time (days) by category (arrival process parameters)
    mean_iat_days: dict[str, float] = field(default_factory=lambda: {
        "stroke": 1.2,
        "tia": 9.3,
        "complex_neuro": 3.6,
        "other": 3.2,
    })

    def __post_init__(self) -> None:
        # Basic validation to catch typos early
        cat_set = set(self.categories)

        if set(self.base_period_counts) != cat_set:
            missing = cat_set - set(self.base_period_counts)
            extra = set(self.base_period_counts) - cat_set
            raise ValueError(f"base_period_counts keys mismatch. missing={missing}, extra={extra}")

        if set(self.mean_iat_days) != cat_set:
            missing = cat_set - set(self.mean_iat_days)
            extra = set(self.mean_iat_days) - cat_set
            raise ValueError(f"mean_iat_days keys mismatch. missing={missing}, extra={extra}")

        for c, m in self.mean_iat_days.items():
            if m <= 0:
                raise ValueError(f"mean_iat_days[{c}] must be > 0")


# -------------------------
# Core data model
# -------------------------

@dataclass(frozen=True)
class Patient:
    id: int
    category: str
    arrival_time: float  # days since t=0


class IdCounter:
    """Simple incrementing ID generator."""
    def __init__(self, start: int = 1) -> None:
        self._next = start

    def next_id(self) -> int:
        val = self._next
        self._next += 1
        return val


# -------------------------
# Arrival processes
# -------------------------

def _arrival_stream(
    env: simpy.Environment,
    *,
    category: str,
    iat_dist: Exponential,
    patients_out: list[Patient],
    id_counter: IdCounter,
    until_days: float,
) -> simpy.events.Event:
    """
    SimPy process: generate arrivals for one category until time >= until_days.
    """
    while True:
        iat = iat_dist.sample()
        yield env.timeout(iat)

        if env.now >= until_days:
            return

        pid = id_counter.next_id()
        patient = Patient(id=pid, category=category, arrival_time=float(env.now))
        patients_out.append(patient)
        trace(f"[t={env.now:8.3f} d] arrival: id={pid:5d} category={category}")


def generate_patients(
    *,
    sim_days: float,
    scenario: Optional[Scenario] = None,
    categories: Optional[Sequence[str]] = None,
) -> list[Patient]:
    """
    Run an arrivals-only simulation and return generated patients.

    Params
    ------
    sim_days:
        Length of simulation (days).
    scenario:
        Scenario container (base case by default).
    categories:
        Which categories to include. Default: all scenario categories.

    Returns
    -------
    List[Patient] with (id, category, arrival_time).
    """
    if sim_days <= 0:
        raise ValueError("sim_days must be > 0")

    sc = scenario or Scenario()
    selected = list(categories) if categories is not None else list(sc.categories)

    unknown = sorted(set(selected) - set(sc.categories))
    if unknown:
        raise ValueError(f"Unknown categories: {unknown}. Valid: {list(sc.categories)}")

    env = simpy.Environment()
    patients: list[Patient] = []
    id_counter = IdCounter(start=1)

    # Master RNG used only to deterministically derive per-stream seeds
    rng_master = np.random.default_rng(sc.random_seed)

    # One exponential IAT distribution instance per category stream
    iat_dists: dict[str, Exponential] = {}
    for cat in selected:
        seed = int(rng_master.integers(0, 2**32 - 1, dtype=np.uint32).item())
        iat_dists[cat] = Exponential(mean=sc.mean_iat_days[cat], random_seed=seed)

    # Start one arrival process per selected category
    for cat in selected:
        env.process(
            _arrival_stream(
                env,
                category=cat,
                iat_dist=iat_dists[cat],
                patients_out=patients,
                id_counter=id_counter,
                until_days=sim_days,
            )
        )

    trace(f"Starting simulation for {sim_days} days; categories={selected}")
    env.run(until=sim_days)
    trace("Simulation finished.")

    # Convenient ordering for downstream pathway modelling
    patients.sort(key=lambda p: p.arrival_time)
    return patients


# -------------------------
# Simple summary helpers
# -------------------------

def summarise_arrivals(patients: list[Patient], scenario: Scenario) -> dict[str, int]:
    counts = {c: 0 for c in scenario.categories}
    for p in patients:
        counts[p.category] += 1
    return counts


if __name__ == "__main__":
    # Demo run
    sc = Scenario(random_seed=42)
    sim_days = 365.0

    patients = generate_patients(sim_days=sim_days, scenario=sc)
    counts = summarise_arrivals(patients, sc)

    total = sum(counts.values()) or 1
    print(f"Simulated {total} arrivals over {sim_days} days")
    for cat in sc.categories:
        print(f"  {cat:14s}: {counts[cat]:5d}  ({counts[cat]/total:6.2%})")

    base_total = sum(sc.base_period_counts.values()) or 1
    print("\nBase-period proportions:")
    for cat in sc.categories:
        print(f"  {cat:14s}: {sc.base_period_counts[cat]/base_total:6.2%}")

Starting simulation for 365.0 days; categories=['stroke', 'tia', 'complex_neuro', 'other']
[t=   0.529 d] arrival: id=    1 category=stroke
[t=   0.865 d] arrival: id=    2 category=other
[t=   1.336 d] arrival: id=    3 category=stroke
[t=   1.740 d] arrival: id=    4 category=other
[t=   1.781 d] arrival: id=    5 category=stroke
[t=   2.120 d] arrival: id=    6 category=stroke
[t=   3.534 d] arrival: id=    7 category=stroke
[t=   3.664 d] arrival: id=    8 category=stroke
[t=   3.686 d] arrival: id=    9 category=stroke
[t=   4.109 d] arrival: id=   10 category=complex_neuro
[t=   4.894 d] arrival: id=   11 category=stroke
[t=   6.336 d] arrival: id=   12 category=complex_neuro
[t=   7.231 d] arrival: id=   13 category=stroke
[t=   7.306 d] arrival: id=   14 category=other
[t=   7.632 d] arrival: id=   15 category=other
[t=   8.313 d] arrival: id=   16 category=complex_neuro
[t=  10.433 d] arrival: id=   17 category=tia
[t=  11.307 d] arrival: id=   18 category=stroke
[t=  11.714 d

## Observations

This code now appear to function as desired. Patient objects are generated at appropriate intervals. These can be passed to the SimPy process and use resources such as acute or rehab beds in the next iteration of development.